# Data Cleaning Pipeline 2v2

Notebook unificado para ejecutar en orden:
1. LimpiezaDatos
2. ColumnDeleter
3. CambiosAVG

Este flujo usa DataFrames en memoria entre secciones y solo guarda el CSV final.

## Section 0 


In [7]:
import os
import re
import sys
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

# =========================
# CONFIG GLOBAL
# =========================
DATASETS_DIR = "datasets"
os.makedirs(DATASETS_DIR, exist_ok=True)

CSV_IN = os.path.join(DATASETS_DIR, "replays_2v2_10000_full.csv")
CSV_OUT = os.path.join(DATASETS_DIR, "replays_subset_with_time_percentages.csv")

THRESHOLD_NULL_RATIO = 0.0001
TREAT_EMPTY_STRINGS_AS_NULL = True
KEEP_COLS = []
SHOW_EXAMPLES = 5
MAX_PRINT_CANDIDATES = 400  # 0 = imprimir todas
DROP_CAMERA_COLUMNS = True
DROP_MATCHES_SHORTER_THAN_SECONDS = 3

# Persistencia de decisiones de la seccion 2.5
DECISIONS_25_FILE = os.path.join(DATASETS_DIR, "section_2_5_decisions.json")

DURATION_CANDIDATES = [
    "duration",
    "duration_seconds",
    "game.duration",
    "game.duration_seconds",
    "replay.duration",
    "replay.duration_seconds",
    "match.duration",
    "match.duration_seconds",
]

print(f"CSV_IN:  {CSV_IN}")
print(f"CSV_OUT: {CSV_OUT}")
print(f"DECISIONS_25_FILE: {DECISIONS_25_FILE}")

CSV_IN:  datasets\replays_2v2_10000_full.csv
CSV_OUT: datasets\replays_subset_with_time_percentages.csv
DECISIONS_25_FILE: datasets\section_2_5_decisions.json


In [8]:
# =========================
# HELPERS COMPARTIDOS
# =========================
def read_csv_safely(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def is_numeric_dtype(dtype) -> bool:
    return pd.api.types.is_numeric_dtype(dtype)


def is_bool_dtype(dtype) -> bool:
    return pd.api.types.is_bool_dtype(dtype)


def shorten(x, n=180):
    s = str(x)
    return s if len(s) <= n else s[:n] + "..."


def print_examples(series: pd.Series, n=5):
    non_null = series.dropna()
    if non_null.empty:
        print("  (no hay valores no nulos para mostrar)")
        return
    for i, v in enumerate(non_null.head(n).tolist(), start=1):
        print(f"  {i}. {shorten(v)}")


def ask_action(col: str, null_pct: float, dtype: str) -> str:
    while True:
        print("\n----------------------------------------")
        print(f"Columna candidata: {col}")
        print(f"  - null_pct: {null_pct:.2f}%")
        print(f"  - dtype: {dtype}")
        ans = input("Accion: [d]=eliminar, [k]=mantener y rellenar nulls, [s]=ver ejemplos, [q]=salir: ").strip().lower()
        if ans in {"d", "k", "s", "q"}:
            return ans
        print("Entrada no valida. Usa d/k/s/q.")


def ask_imputation_numeric(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna numerica):")
        print("  [1] media")
        print("  [2] mediana")
        print("  [3] moda (valor mas frecuente)")
        print("  [4] constante (lo indicas tu)")
        print("  [5] cero")
        print("  [6] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4/5/6: ").strip()

        s = series.dropna()
        if choice == "1":
            if s.empty:
                print("No hay datos no nulos para calcular media. Elige constante.")
                continue
            return ("mean", float(s.mean()))
        if choice == "2":
            if s.empty:
                print("No hay datos no nulos para calcular mediana. Elige constante.")
                continue
            return ("median", float(s.median()))
        if choice == "3":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige constante.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige constante.")
                continue
            try:
                return ("mode", float(mode.iloc[0]))
            except Exception:
                return ("mode", mode.iloc[0])
        if choice == "4":
            raw = input(f"Valor constante para '{col}' (ej: 0, -1, 3.14): ").strip()
            try:
                val = float(raw)
            except ValueError:
                print("Ese valor no parece numerico. Intenta de nuevo.")
                continue
            return ("constant", val)
        if choice == "5":
            return ("zero", 0.0)
        if choice == "6":
            return ("none", None)
        print("Opcion no valida.")


def ask_imputation_bool(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna booleana):")
        print("  [1] True")
        print("  [2] False")
        print("  [3] moda (valor mas frecuente)")
        print("  [4] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4: ").strip()

        s = series.dropna()
        if choice == "1":
            return ("true", True)
        if choice == "2":
            return ("false", False)
        if choice == "3":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige True/False.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige True/False.")
                continue
            return ("mode", bool(mode.iloc[0]))
        if choice == "4":
            return ("none", None)
        print("Opcion no valida.")


def ask_imputation_categorical(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna no numerica):")
        print("  [1] moda (valor mas frecuente)")
        print("  [2] constante (lo indicas tu, texto)")
        print("  [3] 'MISSING' (texto literal)")
        print("  [4] cadena vacia ''")
        print("  [5] boolean True")
        print("  [6] boolean False")
        print("  [7] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4/5/6/7: ").strip()

        s = series.dropna()
        if choice == "1":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige constante.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige constante.")
                continue
            return ("mode", str(mode.iloc[0]))
        if choice == "2":
            val = input(f"Valor constante (texto) para '{col}': ").strip()
            return ("constant", val)
        if choice == "3":
            return ("missing_literal", "MISSING")
        if choice == "4":
            return ("empty_string", "")
        if choice == "5":
            return ("true", True)
        if choice == "6":
            return ("false", False)
        if choice == "7":
            return ("none", None)
        print("Opcion no valida.")


def _to_python_scalar(value):
    if value is None:
        return None
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def load_decisions(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        return {}
    return data


def save_decisions(path: str, decisions: dict) -> None:
    serializable = {}
    for col, payload in decisions.items():
        if not isinstance(payload, dict):
            continue
        action = payload.get("action")
        if action == "drop":
            serializable[col] = {"action": "drop"}
        elif action == "impute":
            serializable[col] = {
                "action": "impute",
                "strategy": payload.get("strategy", "none"),
                "value": _to_python_scalar(payload.get("value")),
            }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=True, indent=2)


def apply_saved_decisions(df_in: pd.DataFrame, decisions: dict) -> tuple[pd.DataFrame, list[str], dict[str, tuple[str, object]]]:
    dropped_cols = []
    imputed_cols = {}
    df_out = df_in.copy()

    for col, payload in decisions.items():
        if col not in df_out.columns:
            continue
        action = payload.get("action")

        if action == "drop":
            dropped_cols.append(col)
            continue

        if action == "impute":
            strat = payload.get("strategy", "none")
            val = payload.get("value")
            if strat != "none":
                df_out[col] = df_out[col].fillna(val)
            preview = shorten(val, 60) if isinstance(val, str) else val
            imputed_cols[col] = (strat, preview)

    if dropped_cols:
        cols_to_drop = [c for c in dropped_cols if c in df_out.columns]
        df_out = df_out.drop(columns=cols_to_drop)

    return df_out, dropped_cols, imputed_cols


def is_camera_col(col: str) -> bool:
    c = col.lower()

    camera_keywords = [
        "camera", "camera_settings", "cam_settings", "camerasettings", "camsettings"
    ]
    if any(k in c for k in camera_keywords):
        return True

    cam_params = ["fov", "distance", "height", "angle", "stiffness", "swivel", "transition"]
    if any(p in c for p in cam_params) and ("player" in c or "players" in c) and ("setting" in c or "settings" in c):
        return True

    return False


def find_duration_column(df: pd.DataFrame) -> str | None:
    for c in DURATION_CANDIDATES:
        if c in df.columns:
            return c
    duration_like = [c for c in df.columns if "duration" in c.lower()]
    if duration_like:
        for c in duration_like:
            if "second" in c.lower():
                return c
        return duration_like[0]
    return None


def rank_name_without_division(value):
    if value is None or pd.isna(value):
        return pd.NA
    s = str(value).strip()
    if not s:
        return pd.NA
    s = re.sub(r"\s+Division\s+\d+\s*$", "", s, flags=re.IGNORECASE)
    return s.strip()


def add_grouped_game_rank_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    mapping = [
        ("min_rank.name", "min.game.rank"),
        ("max_rank.name", "max.game.rank"),
    ]

    for src_col, dst_col in mapping:
        if src_col not in df_out.columns:
            print(f"[WARN] No existe columna fuente para rank agrupado: {src_col}")
            continue
        df_out[dst_col] = df_out[src_col].apply(rank_name_without_division).astype("string")

    return df_out

## Section 1 - Limpieza automatica (en memoria)

Esta seccion carga el CSV inicial y aplica limpieza automatica (camara y duracion).

In [9]:
if not os.path.exists(CSV_IN):
    raise FileNotFoundError(f"No existe el archivo: {CSV_IN}")

df = read_csv_safely(CSV_IN)

if TREAT_EMPTY_STRINGS_AS_NULL:
    df = df.replace(r"^\s*$", pd.NA, regex=True)

rows_before_all = len(df)
cols_before_all = df.shape[1]

print("========================================")
print("Carga inicial")
print("========================================")
print(f"Filas iniciales:    {rows_before_all:,}")
print(f"Columnas iniciales: {cols_before_all:,}")

Carga inicial
Filas iniciales:    10,000
Columnas iniciales: 561


In [10]:
# 0) Eliminacion automatica de columnas de camara
if DROP_CAMERA_COLUMNS:
    camera_cols = [c for c in df.columns if is_camera_col(c)]
    print("\n========================================")
    print("Eliminacion automatica: columnas de camara")
    print("========================================")
    if camera_cols:
        print(f"Columnas detectadas: {len(camera_cols):,}")
        for c in camera_cols:
            print(f"- {c}")
        df = df.drop(columns=camera_cols)
    else:
        print("No se detectaron columnas de camara con el patron actual.")

# 1) Eliminacion automatica de partidas cortas
if DROP_MATCHES_SHORTER_THAN_SECONDS is not None:
    print("\n========================================")
    print(f"Eliminacion automatica: partidas < {DROP_MATCHES_SHORTER_THAN_SECONDS}s")
    print("========================================")
    dur_col = find_duration_column(df)
    if dur_col is None:
        print("No se encontro columna de duracion. No se eliminan filas por duracion.")
        duration_like = [c for c in df.columns if "duration" in c.lower()]
        if duration_like:
            print("Columnas con 'duration':")
            for c in duration_like[:50]:
                print(f"- {c}")
            if len(duration_like) > 50:
                print(f"... ({len(duration_like) - 50} mas)")
        else:
            print("(ninguna)")
    else:
        dur = pd.to_numeric(df[dur_col], errors="coerce")
        mask_short = dur.notna() & (dur < DROP_MATCHES_SHORTER_THAN_SECONDS)
        removed_short = int(mask_short.sum())
        df = df.loc[~mask_short].copy()

        print(f"Columna usada: {dur_col}")
        print(f"Filas eliminadas por duracion: {removed_short:,}")
        print(f"Filas restantes: {len(df):,}")

print("\n========================================")
print("Section 1 completada (limpieza automatica)")
print("========================================")
print(f"Filas tras limpieza automatica:    {len(df):,}")
print(f"Columnas tras limpieza automatica: {df.shape[1]:,}")

df_stage1 = df.copy()


Eliminacion automatica: columnas de camara
Columnas detectadas: 28
- blue.players.0.camera.distance
- blue.players.0.camera.fov
- blue.players.0.camera.height
- blue.players.0.camera.pitch
- blue.players.0.camera.stiffness
- blue.players.0.camera.swivel_speed
- blue.players.0.camera.transition_speed
- blue.players.1.camera.distance
- blue.players.1.camera.fov
- blue.players.1.camera.height
- blue.players.1.camera.pitch
- blue.players.1.camera.stiffness
- blue.players.1.camera.swivel_speed
- blue.players.1.camera.transition_speed
- orange.players.0.camera.distance
- orange.players.0.camera.fov
- orange.players.0.camera.height
- orange.players.0.camera.pitch
- orange.players.0.camera.stiffness
- orange.players.0.camera.swivel_speed
- orange.players.0.camera.transition_speed
- orange.players.1.camera.distance
- orange.players.1.camera.fov
- orange.players.1.camera.height
- orange.players.1.camera.pitch
- orange.players.1.camera.stiffness
- orange.players.1.camera.swivel_speed
- orange.pl

## Section 2 - ColumnDeleter (en memoria)

Primero se aplica la lista corregida de columnas para reducir trabajo innecesario antes de la imputacion interactiva.

In [11]:
def build_keep_columns() -> list[str]:
    keep = set()

    keep.update([
        "duration",
        "overtime",
        "overtime_seconds",
        "max.game.rank",
        "server.region",
        "blue",
        "orange",
        "blue.color",
        "orange.color",
        "blue.players",
        "orange.players",
        "blue.stats",
        "orange.stats",
        "blue.stats.ball",
        "orange.stats.ball",
        "blue.stats.core",
        "orange.stats.core",
        "blue.stats.boost",
        "orange.stats.boost",
        "blue.stats.movement",
        "orange.stats.movement",
        "blue.stats.positioning",
        "orange.stats.positioning",
        "blue.stats.demo",
        "orange.stats.demo",
    ])

    teams = ["blue", "orange"]
    team_ball = ["possession_time", "time_in_side"]
    for t in teams:
        for f in team_ball:
            keep.add(f"{t}.stats.ball.{f}")

    team_core = [
        "shots", "shots_against", "goals", "goals_against", "saves", "assists", "score", "shooting_percentage"
    ]

    team_boost = [
        "bpm", "bcpm", "avg_amount",
        "amount_collected", "amount_stolen",
        "amount_collected_big", "amount_stolen_big",
        "amount_collected_small", "amount_stolen_small",
        "count_collected_big", "count_stolen_big",
        "count_collected_small", "count_stolen_small",
        "amount_overfill", "amount_overfill_stolen",
        "amount_used_while_supersonic",
        "time_zero_boost", "time_full_boost",
        "time_boost_0_25", "time_boost_25_50", "time_boost_50_75", "time_boost_75_100",
    ]

    team_movement = [
        "total_distance",
        "time_supersonic_speed", "time_boost_speed", "time_slow_speed",
        "time_ground", "time_low_air", "time_high_air",
        "time_powerslide", "count_powerslide",
    ]

    team_positioning = [
        "time_defensive_third",
        "time_neutral_third",
        "time_offensive_third",
        "time_defensive_half",
        "time_offensive_half",
        "time_behind_ball",
        "time_infront_ball",
    ]

    team_demo = ["inflicted", "taken"]

    for t in teams:
        for f in team_core:
            keep.add(f"{t}.stats.core.{f}")
        for f in team_boost:
            keep.add(f"{t}.stats.boost.{f}")
        for f in team_movement:
            keep.add(f"{t}.stats.movement.{f}")
        for f in team_positioning:
            keep.add(f"{t}.stats.positioning.{f}")
        for f in team_demo:
            keep.add(f"{t}.stats.demo.{f}")

    return sorted(keep)


if "df_stage1" in globals():
    df = df_stage1.copy()
elif "df" in globals():
    print("[WARN] df_stage1 no existe. Usando el DataFrame actual 'df'.")
    df = df.copy()
else:
    raise NameError("df_stage1 no esta definido. Ejecuta la celda de Section 1 antes de Section 2.")
df = add_grouped_game_rank_columns(df)
keep_cols = build_keep_columns()

existing = [c for c in keep_cols if c in df.columns]
existing = [
    c for c in existing
    if ".players." not in c
    or (".id" not in c and not c.endswith(".name"))
]
missing = [c for c in keep_cols if c not in df.columns]

print("========================================")
print("Section 2 - Seleccion de columnas")
print("========================================")
print(f"Filas actuales:      {len(df):,}")
print(f"Columnas actuales:   {df.shape[1]:,}")
print(f"Cols solicitadas:    {len(keep_cols):,}")
print(f"Cols encontradas:    {len(existing):,}")
print(f"Cols faltantes:      {len(missing):,}")

if missing:
    print("\nColumnas solicitadas que no existen:")
    for c in missing:
        print(f"- {c}")

df = df[existing].copy()
df_stage2 = df.copy()

print(f"\nColumnas tras filtro: {df.shape[1]:,}")

Section 2 - Seleccion de columnas
Filas actuales:      10,000
Columnas actuales:   535
Cols solicitadas:    125
Cols encontradas:    107
Cols faltantes:      18

Columnas solicitadas que no existen:
- blue
- blue.players
- blue.stats
- blue.stats.ball
- blue.stats.boost
- blue.stats.core
- blue.stats.demo
- blue.stats.movement
- blue.stats.positioning
- orange
- orange.players
- orange.stats
- orange.stats.ball
- orange.stats.boost
- orange.stats.core
- orange.stats.demo
- orange.stats.movement
- orange.stats.positioning

Columnas tras filtro: 107


## Section 2.5 - Imputacion interactiva tras filtro

La imputacion interactiva se aplica despues de filtrar columnas para reducir entradas manuales innecesarias.

Esta seccion tambien guarda tus decisiones y, si ejecutas de nuevo, te permite preservar las decisiones anteriores o decidir otra vez.

In [12]:
df = df_stage2.copy()

# Robustez: permite ejecutar esta celda incluso si no se han recargado helpers/config
if "DECISIONS_25_FILE" not in globals():
    DECISIONS_25_FILE = "section_2_5_decisions.json"

if "_to_python_scalar" not in globals():
    def _to_python_scalar(value):
        if value is None:
            return None
        if pd.isna(value):
            return None
        if isinstance(value, np.generic):
            return value.item()
        return value

if "load_decisions" not in globals():
    def load_decisions(path: str) -> dict:
        if not os.path.exists(path):
            return {}
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return data if isinstance(data, dict) else {}
        except Exception:
            print(f"[WARN] No se pudo leer {path}. Se usaran decisiones vacias.")
            return {}

if "save_decisions" not in globals():
    def save_decisions(path: str, decisions: dict) -> None:
        serializable = {}
        for col, payload in decisions.items():
            if not isinstance(payload, dict):
                continue
            action = payload.get("action")
            if action == "drop":
                serializable[col] = {"action": "drop"}
            elif action == "impute":
                serializable[col] = {
                    "action": "impute",
                    "strategy": payload.get("strategy", "none"),
                    "value": _to_python_scalar(payload.get("value")),
                }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(serializable, f, ensure_ascii=True, indent=2)

if "apply_saved_decisions" not in globals():
    def apply_saved_decisions(df_in: pd.DataFrame, decisions: dict) -> tuple[pd.DataFrame, list[str], dict[str, tuple[str, object]]]:
        dropped_cols = []
        imputed_cols = {}
        df_out = df_in.copy()

        for col, payload in decisions.items():
            if col not in df_out.columns:
                continue
            action = payload.get("action")

            if action == "drop":
                dropped_cols.append(col)
                continue

            if action == "impute":
                strat = payload.get("strategy", "none")
                val = payload.get("value")
                if strat != "none":
                    df_out[col] = df_out[col].fillna(val)
                preview = shorten(val, 60) if isinstance(val, str) else val
                imputed_cols[col] = (strat, preview)

        if dropped_cols:
            cols_to_drop = [c for c in dropped_cols if c in df_out.columns]
            df_out = df_out.drop(columns=cols_to_drop)

        return df_out, dropped_cols, imputed_cols

saved_decisions = load_decisions(DECISIONS_25_FILE)
use_saved = False

if saved_decisions:
    print("\n========================================")
    print("Decisiones previas detectadas")
    print("========================================")
    print(f"Archivo: {DECISIONS_25_FILE}")
    print(f"Columnas con decision guardada: {len(saved_decisions):,}")

    while True:
        choice = input("Quieres preservar decisiones previas o decidir otra vez? [p]=preservar, [r]=repetir: ").strip().lower()
        if choice in {"p", "r"}:
            use_saved = (choice == "p")
            break
        print("Entrada no valida. Usa p/r.")

if use_saved:
    df, dropped_cols, imputed_cols = apply_saved_decisions(df, saved_decisions)
    decisions = saved_decisions
    print("\nSe aplicaron las decisiones guardadas.")
else:
    null_ratio = df.isna().mean()
    keep_set = set(KEEP_COLS)

    candidates = [c for c in df.columns if (null_ratio[c] > THRESHOLD_NULL_RATIO and c not in keep_set)]
    candidates.sort(key=lambda c: null_ratio[c], reverse=True)

    dropped_cols = []
    imputed_cols = {}
    decisions = {}

    print("\n========================================")
    print("Candidatas a revisar (muchos nulos)")
    print(f"Total candidatas (null_ratio > {THRESHOLD_NULL_RATIO}): {len(candidates):,}")
    print("Formato: columna | null_pct | dtype")
    print("========================================")

    if not candidates:
        print("No hay columnas candidatas. Se continua a la siguiente seccion.")
    else:
        to_show = candidates if MAX_PRINT_CANDIDATES == 0 else candidates[:MAX_PRINT_CANDIDATES]
        for col in to_show:
            print(f"{col} | {null_ratio[col]*100:.2f}% | {df[col].dtype}")
        if MAX_PRINT_CANDIDATES != 0 and len(candidates) > MAX_PRINT_CANDIDATES:
            print(f"... ({len(candidates) - MAX_PRINT_CANDIDATES} mas no mostradas)")

        input("\nPulsa ENTER para empezar a decidir columna por columna...")

        for idx, col in enumerate(candidates, start=1):
            pct = float(null_ratio[col] * 100.0)
            dtype = str(df[col].dtype)

            print(f"\n[{idx}/{len(candidates)}]")

            while True:
                action = ask_action(col, pct, dtype)

                if action == "s":
                    print("\nEjemplos (no nulos):")
                    print_examples(df[col], n=SHOW_EXAMPLES)
                    continue

                if action == "q":
                    print("Salida solicitada. No se continua la ejecucion.")
                    raise SystemExit(0)

                if action == "d":
                    dropped_cols.append(col)
                    decisions[col] = {"action": "drop"}
                    break

                if action == "k":
                    if is_bool_dtype(df[col].dtype):
                        strat, val = ask_imputation_bool(col, df[col])
                    elif is_numeric_dtype(df[col].dtype):
                        strat, val = ask_imputation_numeric(col, df[col])
                    else:
                        strat, val = ask_imputation_categorical(col, df[col].astype("string"))

                    if strat != "none":
                        df[col] = df[col].fillna(val)

                    preview = val
                    if isinstance(preview, str):
                        preview = shorten(preview, 60)
                    imputed_cols[col] = (strat, preview)
                    decisions[col] = {
                        "action": "impute",
                        "strategy": strat,
                        "value": _to_python_scalar(val),
                    }
                    break

    if dropped_cols:
        df = df.drop(columns=dropped_cols)

    save_decisions(DECISIONS_25_FILE, decisions)
    print(f"\nDecisiones guardadas en: {DECISIONS_25_FILE}")

df_stage2_clean = df.copy()

print("\n========================================")
print("Section 2.5 completada (imputacion interactiva)")
print("========================================")
print(f"Filas actuales: {len(df):,}")
print(f"Columnas actuales: {df.shape[1]:,}")
print(f"Columnas eliminadas (interactivo): {len(dropped_cols):,}")
print(f"Columnas imputadas (rellenadas): {len(imputed_cols):,}")
print(f"Total nulls restantes: {int(df.isna().sum().sum()):,}")


Decisiones previas detectadas
Archivo: datasets\section_2_5_decisions.json
Columnas con decision guardada: 5

Se aplicaron las decisiones guardadas.

Section 2.5 completada (imputacion interactiva)
Filas actuales: 10,000
Columnas actuales: 107
Columnas eliminadas (interactivo): 0
Columnas imputadas (rellenadas): 5
Total nulls restantes: 0


## Section 3 - CambiosAVG (en memoria)

Se crean columnas *_pct a partir de columnas de tiempo, usando duration como denominador.

In [13]:
DURATION_COL = "duration"
TEAM_TIME_COLS = [
    "blue.stats.ball.possession_time",
    "blue.stats.ball.time_in_side",
    "blue.stats.boost.time_zero_boost",
    "blue.stats.boost.time_full_boost",
    "blue.stats.boost.time_boost_0_25",
    "blue.stats.boost.time_boost_25_50",
    "blue.stats.boost.time_boost_50_75",
    "blue.stats.boost.time_boost_75_100",
    "blue.stats.movement.time_supersonic_speed",
    "blue.stats.movement.time_boost_speed",
    "blue.stats.movement.time_slow_speed",
    "blue.stats.movement.time_ground",
    "blue.stats.movement.time_low_air",
    "blue.stats.movement.time_high_air",
    "blue.stats.movement.time_powerslide",
    "blue.stats.positioning.time_defensive_third",
    "blue.stats.positioning.time_neutral_third",
    "blue.stats.positioning.time_offensive_third",
    "blue.stats.positioning.time_defensive_half",
    "blue.stats.positioning.time_offensive_half",
    "blue.stats.positioning.time_behind_ball",
    "blue.stats.positioning.time_infront_ball",
    "orange.stats.ball.possession_time",
    "orange.stats.ball.time_in_side",
    "orange.stats.boost.time_zero_boost",
    "orange.stats.boost.time_full_boost",
    "orange.stats.boost.time_boost_0_25",
    "orange.stats.boost.time_boost_25_50",
    "orange.stats.boost.time_boost_50_75",
    "orange.stats.boost.time_boost_75_100",
    "orange.stats.movement.time_supersonic_speed",
    "orange.stats.movement.time_boost_speed",
    "orange.stats.movement.time_slow_speed",
    "orange.stats.movement.time_ground",
    "orange.stats.movement.time_low_air",
    "orange.stats.movement.time_high_air",
    "orange.stats.movement.time_powerslide",
    "orange.stats.positioning.time_defensive_third",
    "orange.stats.positioning.time_neutral_third",
    "orange.stats.positioning.time_offensive_third",
    "orange.stats.positioning.time_defensive_half",
    "orange.stats.positioning.time_offensive_half",
    "orange.stats.positioning.time_behind_ball",
    "orange.stats.positioning.time_infront_ball",
]


def add_pct_column(df_in: pd.DataFrame, time_col: str, duration_col: str) -> bool:
    if time_col not in df_in.columns:
        print(f"[WARN] No existe: {time_col}")
        return False
    if duration_col not in df_in.columns:
        raise KeyError(f"No existe la columna de duracion: {duration_col}")

    t = to_numeric(df_in[time_col])
    d = to_numeric(df_in[duration_col])

    pct = (t / d) * 100.0
    pct = pct.where(d > 0)

    # Reemplaza la columna original de tiempo por su porcentaje sobre duracion
    df_in[time_col] = pct
    return True


df = df_stage2_clean.copy()
converted_pct_cols = []

for col in TEAM_TIME_COLS:
    if add_pct_column(df, col, DURATION_COL):
        converted_pct_cols.append(col)

df_stage3 = df.copy()

print("========================================")
print("Section 3 - Conversion a porcentajes (in-place)")
print("========================================")
print(f"Columnas convertidas a %: {len(converted_pct_cols):,}")
if converted_pct_cols:
    print("Primeras 10 columnas convertidas:")
    for c in converted_pct_cols[:10]:
        print(f"- {c}")

Section 3 - Conversion a porcentajes (in-place)
Columnas convertidas a %: 44
Primeras 10 columnas convertidas:
- blue.stats.ball.possession_time
- blue.stats.ball.time_in_side
- blue.stats.boost.time_zero_boost
- blue.stats.boost.time_full_boost
- blue.stats.boost.time_boost_0_25
- blue.stats.boost.time_boost_25_50
- blue.stats.boost.time_boost_50_75
- blue.stats.boost.time_boost_75_100
- blue.stats.movement.time_supersonic_speed
- blue.stats.movement.time_boost_speed


## Final - Validacion y export

Solo se guarda el CSV final del pipeline.

In [14]:
df_final = df_stage3.copy()

if "converted_pct_cols" in globals():
    pct_cols = [c for c in converted_pct_cols if c in df_final.columns]
else:
    pct_cols = [c for c in df_final.columns if c.endswith("_pct")]

print("========================================")
print("Resumen final")
print("========================================")
print(f"Filas finales:      {len(df_final):,}")
print(f"Columnas finales:   {df_final.shape[1]:,}")
print(f"Total nulls final:  {int(df_final.isna().sum().sum()):,}")
print(f"Columnas en porcentaje: {len(pct_cols):,}")

if pct_cols:
    desc = df_final[pct_cols].describe().T[["mean", "min", "max"]]
    print("\nResumen de columnas en porcentaje (primeras 15):")
    display(desc.head(15))

# Unica exportacion del flujo unificado
print(f"\nGuardando CSV final en: {CSV_OUT}")
df_final.to_csv(CSV_OUT, index=False)

print("\nProceso completado.")

Resumen final
Filas finales:      10,000
Columnas finales:   107
Total nulls final:  0
Columnas en porcentaje: 44

Resumen de columnas en porcentaje (primeras 15):


,mean,min,max
blue.stats.ball.possession_time,37.540408,0.0,66.601562
blue.stats.ball.time_in_side,45.519349,0.0,113.500000
blue.stats.boost.time_zero_boost,23.117383,0.0,75.753843
blue.stats.boost.time_full_boost,21.868240,0.0,82.200000
blue.stats.boost.time_boost_0_25,62.552993,0.0,129.000009
blue.stats.boost.time_boost_25_50,42.664745,0.0,92.625000
blue.stats.boost.time_boost_50_75,33.373712,0.0,67.457143
blue.stats.boost.time_boost_75_100,51.097189,0.0,109.945455
blue.stats.movement.time_supersonic_speed,26.917223,0.0,69.923077
blue.stats.movement.time_boost_speed,75.815610,0.0,102.164773



Guardando CSV final en: datasets\replays_subset_with_time_percentages.csv

Proceso completado.


# Probando cosas

In [1]:
import pandas as pd
import os

DATASETS_DIR = "datasets"
CSV_FINAL = os.path.join(DATASETS_DIR, "replays_subset_with_time_percentages.csv")

df_model = pd.read_csv(CSV_FINAL, low_memory=False)

print("Shape del dataset final:", df_model.shape)
display(df_model.head())

Shape del dataset final: (10000, 107)


,blue.color,blue.stats.ball.possession_time,blue.stats.ball.time_in_side,blue.stats.boost.amount_collected,blue.stats.boost.amount_collected_big,blue.stats.boost.amount_collected_small,blue.stats.boost.amount_overfill,blue.stats.boost.amount_overfill_stolen,blue.stats.boost.amount_stolen,blue.stats.boost.amount_stolen_big,...,orange.stats.positioning.time_behind_ball,orange.stats.positioning.time_defensive_half,orange.stats.positioning.time_defensive_third,orange.stats.positioning.time_infront_ball,orange.stats.positioning.time_neutral_third,orange.stats.positioning.time_offensive_half,orange.stats.positioning.time_offensive_third,overtime,overtime_seconds,server.region
0,blue,44.282927,39.068293,2506,2110,396,289,30,516,369,...,162.912195,128.600000,96.702434,34.546341,57.858537,68.853659,42.887805,False,0.0,USE
1,blue,40.938907,69.038585,3835,2788,1047,588,76,663,427,...,162.025730,106.874598,66.543408,36.411576,82.903543,91.562701,48.987138,False,0.0,USE
2,blue,31.317919,43.644509,5306,3816,1490,403,32,1103,654,...,143.297688,135.034682,97.141618,56.421965,68.057803,64.684971,34.523121,False,0.0,USE
3,blue,36.722500,36.790000,4858,3752,1106,618,251,1148,868,...,150.320000,134.945000,107.170000,47.282500,55.260002,62.655000,35.170000,False,0.0,EU
4,blue,38.622642,49.783019,1281,1006,275,405,38,188,167,...,171.433962,119.905666,83.575472,26.509434,73.584906,78.037736,40.783019,False,0.0,EU


In [2]:
print("\nTipos de datos:")
print(df_model.dtypes)

print("\nNulos por columna:")
print(df_model.isna().sum().sort_values(ascending=False).head(20))

print("\nDistribucion del target:")
print(df_model["max.game.rank"].value_counts(dropna=False))


Tipos de datos:
blue.color                                        object
blue.stats.ball.possession_time                  float64
blue.stats.ball.time_in_side                     float64
blue.stats.boost.amount_collected                  int64
blue.stats.boost.amount_collected_big              int64
                                                  ...   
orange.stats.positioning.time_offensive_half     float64
orange.stats.positioning.time_offensive_third    float64
overtime                                            bool
overtime_seconds                                 float64
server.region                                     object
Length: 107, dtype: object

Nulos por columna:
blue.color                                         0
orange.stats.core.goals_against                    0
orange.stats.core.assists                          0
orange.stats.boost.time_zero_boost                 0
orange.stats.boost.time_full_boost                 0
orange.stats.boost.time_boost_75_100        

In [3]:
target_col = "max.game.rank"

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (10000, 106)
Shape de y: (10000,)


In [4]:
categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Columnas categoricas:", categorical_cols)
print("Numero de categoricas:", len(categorical_cols))

print("\nNumero de numericas/booleanas:", len(numeric_cols))
print("Primeras 20 numericas:", numeric_cols[:20])

Columnas categoricas: ['blue.color', 'orange.color', 'server.region']
Numero de categoricas: 3

Numero de numericas/booleanas: 103
Primeras 20 numericas: ['blue.stats.ball.possession_time', 'blue.stats.ball.time_in_side', 'blue.stats.boost.amount_collected', 'blue.stats.boost.amount_collected_big', 'blue.stats.boost.amount_collected_small', 'blue.stats.boost.amount_overfill', 'blue.stats.boost.amount_overfill_stolen', 'blue.stats.boost.amount_stolen', 'blue.stats.boost.amount_stolen_big', 'blue.stats.boost.amount_stolen_small', 'blue.stats.boost.amount_used_while_supersonic', 'blue.stats.boost.avg_amount', 'blue.stats.boost.bcpm', 'blue.stats.boost.bpm', 'blue.stats.boost.count_collected_big', 'blue.stats.boost.count_collected_small', 'blue.stats.boost.count_stolen_big', 'blue.stats.boost.count_stolen_small', 'blue.stats.boost.time_boost_0_25', 'blue.stats.boost.time_boost_25_50']


In [5]:
class_counts = y.value_counts()
print(class_counts)

print("\nNumero de clases:", y.nunique())
print("Clase mayoritaria:", class_counts.max())
print("Clase minoritaria:", class_counts.min())
print("Imbalance ratio:", class_counts.max() / class_counts.min())

max.game.rank
Champion III          1632
Grand Champion I      1626
Champion II           1517
Grand Champion II     1505
Champion I            1202
Grand Champion III    1079
Diamond III            542
Diamond II             411
Diamond I              243
Platinum III           111
Platinum II             44
Platinum I              40
Gold III                19
Gold II                 15
Gold I                  11
Silver III               3
Name: count, dtype: int64

Numero de clases: 16
Clase mayoritaria: 1632
Clase minoritaria: 3
Imbalance ratio: 544.0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nDistribucion train:")
print(y_train.value_counts(normalize=True).head())

print("\nDistribucion test:")
print(y_test.value_counts(normalize=True).head())

Train shape: (8000, 106)
Test shape: (2000, 106)

Distribucion train:
max.game.rank
Champion III         0.163250
Grand Champion I     0.162625
Champion II          0.151625
Grand Champion II    0.150500
Champion I           0.120250
Name: proportion, dtype: float64

Distribucion test:
max.game.rank
Champion III         0.1630
Grand Champion I     0.1625
Champion II          0.1520
Grand Champion II    0.1505
Champion I           0.1200
Name: proportion, dtype: float64


In [8]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, matthews_corrcoef

# =========================
# 1. CARGAR CSV FINAL
# =========================
DATASETS_DIR = "datasets"
CSV_FINAL = os.path.join(DATASETS_DIR, "replays_subset_with_time_percentages.csv")

df_model = pd.read_csv(CSV_FINAL, low_memory=False)

print("========================================")
print("Dataset cargado")
print("========================================")
print("Shape original:", df_model.shape)
display(df_model.head())

# =========================
# 2. LIMPIEZA MINIMA PARA MODELADO
# =========================
# Quitamos columnas categoricas que no aportan informacion real
cols_to_drop = ["blue.color", "orange.color"]
cols_to_drop = [c for c in cols_to_drop if c in df_model.columns]

if cols_to_drop:
    df_model = df_model.drop(columns=cols_to_drop)

print("\n========================================")
print("Columnas eliminadas antes de modelar")
print("========================================")
print(cols_to_drop if cols_to_drop else "No se elimino ninguna columna extra")

# =========================
# 3. DEFINIR TARGET Y FILTRAR CLASES RARAS
# =========================
target_col = "max.game.rank"

if target_col not in df_model.columns:
    raise KeyError(f"No existe la columna target: {target_col}")

print("\n========================================")
print("Distribucion original del target")
print("========================================")
original_counts = df_model[target_col].value_counts().sort_values()
display(original_counts)

# Cambia este valor si quieres ser mas estricto
# Con 5 ya puedes usar StratifiedKFold de 5 folds sin warning
MIN_CLASS_COUNT = 5

valid_classes = original_counts[original_counts >= MIN_CLASS_COUNT].index
removed_classes = original_counts[original_counts < MIN_CLASS_COUNT]

df_model_filtered = df_model[df_model[target_col].isin(valid_classes)].copy()

print("\n========================================")
print("Filtrado de clases raras")
print("========================================")
print(f"Minimo de ejemplos por clase exigido: {MIN_CLASS_COUNT}")
print(f"Numero de clases originales: {df_model[target_col].nunique()}")
print(f"Numero de clases tras filtrado: {df_model_filtered[target_col].nunique()}")
print(f"Filas originales: {len(df_model):,}")
print(f"Filas tras filtrado: {len(df_model_filtered):,}")
print(f"Filas eliminadas: {len(df_model) - len(df_model_filtered):,}")

print("\nClases eliminadas por ser demasiado raras:")
display(removed_classes)

print("\nDistribucion del target tras filtrado:")
filtered_counts = df_model_filtered[target_col].value_counts().sort_values()
display(filtered_counts)

# =========================
# 4. DEFINIR X E y
# =========================
X = df_model_filtered.drop(columns=[target_col])
y = df_model_filtered[target_col]

print("\n========================================")
print("Definicion de X e y")
print("========================================")
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

# =========================
# 5. IDENTIFICAR COLUMNAS NUMERICAS Y CATEGORICAS
# =========================
categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("\n========================================")
print("Tipos de columnas")
print("========================================")
print("Columnas categoricas:", categorical_cols)
print("Numero de categoricas:", len(categorical_cols))
print("Numero de numericas/booleanas:", len(numeric_cols))

# =========================
# 6. TRAIN / TEST SPLIT ESTRATIFICADO
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n========================================")
print("Train / Test split")
print("========================================")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

print("\nDistribucion de clases en train:")
display(y_train.value_counts().sort_values())

print("\nDistribucion de clases en test:")
display(y_test.value_counts().sort_values())

# =========================
# 7. PREPROCESSING
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================
# 8. VALIDACION CRUZADA Y METRICAS
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "mcc": make_scorer(matthews_corrcoef)
}

# =========================
# 9. BASELINE DUMMY
# =========================
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

dummy_scores = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

dummy_results = pd.DataFrame(dummy_scores)

print("\n========================================")
print("Dummy baseline results (CV)")
print("========================================")
display(dummy_results)

print("\nMean scores:")
for metric in scoring.keys():
    print(f"{metric}: {dummy_results[f'test_{metric}'].mean():.4f}")

Dataset cargado
Shape original: (10000, 107)


,blue.color,blue.stats.ball.possession_time,blue.stats.ball.time_in_side,blue.stats.boost.amount_collected,blue.stats.boost.amount_collected_big,blue.stats.boost.amount_collected_small,blue.stats.boost.amount_overfill,blue.stats.boost.amount_overfill_stolen,blue.stats.boost.amount_stolen,blue.stats.boost.amount_stolen_big,...,orange.stats.positioning.time_behind_ball,orange.stats.positioning.time_defensive_half,orange.stats.positioning.time_defensive_third,orange.stats.positioning.time_infront_ball,orange.stats.positioning.time_neutral_third,orange.stats.positioning.time_offensive_half,orange.stats.positioning.time_offensive_third,overtime,overtime_seconds,server.region
0,blue,44.282927,39.068293,2506,2110,396,289,30,516,369,...,162.912195,128.600000,96.702434,34.546341,57.858537,68.853659,42.887805,False,0.0,USE
1,blue,40.938907,69.038585,3835,2788,1047,588,76,663,427,...,162.025730,106.874598,66.543408,36.411576,82.903543,91.562701,48.987138,False,0.0,USE
2,blue,31.317919,43.644509,5306,3816,1490,403,32,1103,654,...,143.297688,135.034682,97.141618,56.421965,68.057803,64.684971,34.523121,False,0.0,USE
3,blue,36.722500,36.790000,4858,3752,1106,618,251,1148,868,...,150.320000,134.945000,107.170000,47.282500,55.260002,62.655000,35.170000,False,0.0,EU
4,blue,38.622642,49.783019,1281,1006,275,405,38,188,167,...,171.433962,119.905666,83.575472,26.509434,73.584906,78.037736,40.783019,False,0.0,EU



Columnas eliminadas antes de modelar
['blue.color', 'orange.color']

Distribucion original del target


max.game.rank
Silver III               3
Gold I                  11
Gold II                 15
Gold III                19
Platinum I              40
Platinum II             44
Platinum III           111
Diamond I              243
Diamond II             411
Diamond III            542
Grand Champion III    1079
Champion I            1202
Grand Champion II     1505
Champion II           1517
Grand Champion I      1626
Champion III          1632
Name: count, dtype: int64


Filtrado de clases raras
Minimo de ejemplos por clase exigido: 5
Numero de clases originales: 16
Numero de clases tras filtrado: 15
Filas originales: 10,000
Filas tras filtrado: 9,997
Filas eliminadas: 3

Clases eliminadas por ser demasiado raras:


...


Distribucion de clases en test:


max.game.rank
Gold I                  2
Gold II                 3
Gold III                4
Platinum I              8
Platinum II             9
Platinum III           22
Diamond I              49
Diamond II             82
Diamond III           108
Grand Champion III    216
Champion I            240
Grand Champion II     301
Champion II           304
Grand Champion I      325
Champion III          327
Name: count, dtype: int64


Dummy baseline results (CV)


,fit_time,score_time,test_accuracy,test_f1_macro,test_f1_weighted,test_precision_macro,test_recall_macro,test_mcc
0,0.209868,0.043481,0.163125,0.01870,0.045756,0.010875,0.066667,0.0
1,0.205870,0.041972,0.163125,0.01870,0.045756,0.010875,0.066667,0.0
2,0.215223,0.056702,0.163227,0.01871,0.045809,0.010882,0.066667,0.0
3,0.223492,0.040381,0.163227,0.01871,0.045809,0.010882,0.066667,0.0
4,0.218228,0.048173,0.163227,0.01871,0.045809,0.010882,0.066667,0.0



Mean scores:
accuracy: 0.1632
f1_macro: 0.0187
f1_weighted: 0.0458
precision_macro: 0.0109
recall_macro: 0.0667
mcc: 0.0000


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd

# =========================
# 10. PRIMERA COMPARACION DE MODELOS REALES
# =========================
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),
    "DecisionTree": DecisionTreeClassifier(
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for model_name, model in models.items():
    print("========================================")
    print(f"Entrenando y evaluando: {model_name}")
    print("========================================")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    row = {"model": model_name}
    for metric in scoring.keys():
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()

    results.append(row)

results_df = pd.DataFrame(results).sort_values("f1_macro_mean", ascending=False)

print("========================================")
print("Resultados - primera comparacion de modelos")
print("========================================")
display(results_df)

Entrenando y evaluando: LogisticRegression
Entrenando y evaluando: DecisionTree
Entrenando y evaluando: RandomForest
Resultados - primera comparacion de modelos


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,f1_weighted_mean,f1_weighted_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,mcc_mean,mcc_std
0,LogisticRegression,0.379393,0.006295,0.293043,0.028871,0.377298,0.006253,0.312254,0.039723,0.287938,0.026594,0.284475,0.006897
2,RandomForest,0.333372,0.010726,0.204148,0.019441,0.323465,0.011469,0.268183,0.016486,0.192900,0.015142,0.224926,0.012664
1,DecisionTree,0.242215,0.008391,0.168515,0.017105,0.242273,0.008161,0.169639,0.016700,0.171143,0.020852,0.128654,0.009780


In [10]:
from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd

# =========================
# ENSSEMBLE MODELS COMPARISON
# =========================
ensemble_models = {
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    ),
    "Voting": VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=3000, random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42)),
            ("rf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
        ],
        voting="soft"
    ),
    "Stacking": StackingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=3000, random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42)),
            ("rf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
        ],
        final_estimator=LogisticRegression(max_iter=3000, random_state=42),
        cv=3,
        n_jobs=-1
    )
}

ensemble_results = []

for model_name, model in ensemble_models.items():
    print(f"Evaluating: {model_name}")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    row = {"model": model_name}
    for metric in scoring.keys():
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()

    ensemble_results.append(row)

ensemble_results_df = pd.DataFrame(ensemble_results).sort_values(
    "f1_macro_mean", ascending=False
)

print("========================================")
print("Ensemble models comparison")
print("========================================")
display(ensemble_results_df)

Evaluating: Bagging
Evaluating: AdaBoost
Evaluating: GradientBoosting
Evaluating: Voting
Evaluating: Stacking
Ensemble models comparison


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,f1_weighted_mean,f1_weighted_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,mcc_mean,mcc_std
4,Stacking,0.397898,0.008054,0.251461,0.020185,0.392950,0.007907,0.268652,0.032524,0.246859,0.016815,0.304402,0.009154
2,GradientBoosting,0.330624,0.006830,0.211866,0.008584,0.327977,0.006992,0.228677,0.014897,0.205033,0.007535,0.226813,0.008367
0,Bagging,0.324994,0.011057,0.198564,0.017556,0.318379,0.011276,0.254253,0.063883,0.188694,0.012842,0.217242,0.013282
3,Voting,0.246592,0.009562,0.169639,0.010694,0.246536,0.009336,0.172563,0.011709,0.169907,0.011102,0.133854,0.011130
1,AdaBoost,0.248346,0.017708,0.147122,0.018946,0.227481,0.018246,0.164269,0.019158,0.154000,0.019149,0.134008,0.021412
